# Museum Art Guide - full rerun on Google Colab

1. `Runtime > Change runtime type > T4 GPU` (the GPU speeds up the ResNet steps; everything else runs on CPU too).
2. Check `REPO` and `BRANCH` in the 3rd cell (default: `main` of the GitHub repository), then `Runtime > Run all`.
3. At the end, download `results.zip` and `checkpoints.zip` and commit the results **from your own computer** (last cell).

The run executes `scripts/run_all.sh`, the same script as on a laptop, so the notebook cannot drift from the pipeline.
Results are written to `/content/run_results`, outside the repository: overwriting the committed `results/` during
the run would mark every later result as produced by modified code (`git_dirty: true` in its provenance).

The WikiArt images stay on Colab / your Drive: they are never committed (licence unknown).

In [ ]:
!nvidia-smi -L || echo "no GPU: fine for steps 0-3, slower for steps 4 and 8"

In [ ]:
REPO = "https://github.com/loldakowski/museum-art-guide.git"
BRANCH = "main"
import os
%cd /content
if not os.path.isdir("museum-art-guide"):
    !git clone -q -b $BRANCH $REPO museum-art-guide
%cd /content/museum-art-guide
!git checkout -q $BRANCH && git pull -q && git log -1 --format="code: %H %s"
!git status --porcelain --untracked-files=no | head && echo "(nothing above = clean working tree)"

## Exact library versions
`requirements-lock.txt` replaces Colab's preinstalled torch with the version used on the laptop and in CI (2.14.1).
Its PyPI build for Linux is a CUDA 13 build, like Colab's own torch (2.11.0+cu130 in the previous run), so the GPU
keeps working; the next cell checks it. pip may warn that some Colab packages want other pandas/numpy versions: the
experiments run in fresh `python` processes, so this does not affect them.

In [ ]:
!pip install -q -r requirements-lock.txt && pip install -q -e .

In [ ]:
import subprocess, json
out = subprocess.run(["python", "-c", "import sys, json, torch, torchvision, sklearn, numpy, pandas, PIL, open_clip; "
    "print(json.dumps({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, "
    "'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'torchvision': torchvision.__version__, "
    "'sklearn': sklearn.__version__, 'numpy': numpy.__version__, 'pandas': pandas.__version__, 'pillow': PIL.__version__, "
    "'open_clip': open_clip.__version__}))"], capture_output=True, text=True, check=True).stdout
env = json.loads(out); print(json.dumps(env, indent=1))
lock = dict(l.split("==") for l in open("requirements-lock.txt").read().split()  if "==" in l)
assert env["torch"].split("+")[0] == lock["torch"], "torch is not the locked version"
has_gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode == 0
assert env["gpu"] or not has_gpu, "a GPU is attached but torch does not see it: do not run, report this"

In [ ]:
!python -m pytest -q

## Data (kept on Google Drive so that a new Colab session does not download it again)
The main subset at 128 px (classification) and 224 px (free-text search), and the named-artist subset (step 8).

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ["wikiart_style5", "wikiart_style5_224", "wikiart_named5"]:
        !mkdir -p /content/drive/MyDrive/museum-art-guide-data/$d
        !ln -sfn /content/drive/MyDrive/museum-art-guide-data/$d data/$d

In [ ]:
# Plan (dry run) then download; re-running resumes. Main subset: two sizes, ~8.4 GB streamed. Named subset: ~3.7 GB.
!bash scripts/build_dataset.sh
!bash scripts/build_named_dataset.sh
!cat data/wikiart_style5/subset_info.json

## Phone photo for step 5 (optional)
Colab's CPU is not a kiosk: the kiosk-like latency is the laptop run (`results/05_benchmark_laptop`). If you upload
the phone photo (`phone_photo.jpg`, the JPEG converted from the iPhone HEIC), step 5 times it here too; otherwise
it uses a synthetic 12-megapixel JPEG.

In [ ]:
UPLOAD_PHOTO = False
if UPLOAD_PHOTO:
    from google.colab import files
    up = files.upload()
    name, photo = next(iter(up.items()))
    if photo[:3] != b"\xff\xd8\xff":   # every JPEG file starts with these bytes; HEIC, PNG... do not
        raise ValueError(f"{name} is not a JPEG: nothing was saved. Upload phone_photo.jpg (an iPhone HEIC must "
                         "first be converted: sips -s format jpeg -s formatOptions 90 IMG.HEIC --out phone_photo.jpg)")
    os.makedirs("data/kiosk", exist_ok=True)
    open("data/kiosk/phone_photo.jpg", "wb").write(photo)
    print(f"{name} ({len(photo) / 1e6:.2f} MB) saved as data/kiosk/phone_photo.jpg")

## Every step: 00 explore, duplicates, 01 baseline, 02 MLP, 03 diagnosis, 04 ResNet, 05 benchmark, 07 search, 06 summary, 08 unseen painters

In [ ]:
%%bash
set -o pipefail   # the cell fails (and Run all stops) if any step fails
rm -rf /content/run_results
RESULTS=/content/run_results bash scripts/run_all.sh 2>&1 | tee /content/run_all.log

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open("/content/run_results/summary.md").read()))
display(Image("/content/run_results/tradeoff.png"))
display(Markdown(open("/content/run_results/08_unseen_painters/summary.md").read()))

In [ ]:
# Which commit produced the results, and was the code clean? (every line should say dirty=False and the same SHA)
import glob
for f in sorted(glob.glob("/content/run_results/**/*.json", recursive=True)):
    if "/runs/" in f: continue   # step 8 per-run files (not committed)
    d = json.load(open(f))
    p = d.get("provenance") if isinstance(d, dict) else None
    if isinstance(p, dict):
        print(p["git_sha"][:7], "dirty=" + str(p["git_dirty"]), f.replace("/content/run_results/", ""))

In [ ]:
# Download. samples.png / errors.png show paintings: use them in the slides, they are git-ignored.
# checkpoints.zip (not committed) is for rerunning step 5 on the laptop with the models of this run.
!cd /content && rm -f results.zip && cd run_results && zip -qr ../results.zip . && cd .. && zip -qj results.zip run_all.log
!cd /content/museum-art-guide && rm -f /content/checkpoints.zip && zip -qr /content/checkpoints.zip checkpoints
from google.colab import files
files.download("/content/results.zip")
files.download("/content/checkpoints.zip")
# Then on your computer: replace the step folders of results/ with the zip's content (keep results/05_benchmark_laptop),
# check `git diff --stat`, commit on a branch.